# PHA - Health Coach Agent Demo

This notebook demonstrates the **Health Coach Agent** from PHA.

The Health Coach Agent is a conversational health assistant that:
- Maintains multi-turn conversation context
- Guides users through health discussions
- Tracks memory states (goals, constraints, profile)
- Provides personalized recommendations

## Features

1. **Conversation Management**: Tracks full conversation history
2. **Memory States**: Remembers user goals, constraints, what they've tried
3. **Smart Flow Control**: Detects when to make recommendations or end conversation
4. **Simple Mode**: Faster responses with lighter memory updates

## Setup

In [ ]:
import os
import sys

# Add the project root to the path
sys.path.insert(0, '..')

# --- Configuration ---
# Set your API key and provider here
API_KEY = "your-api-key-here"
# Provider options: "gemini", "openai", "anthropic"
PROVIDER = "gemini"


In [ ]:
from pha.agents import HealthCoachAgent, create_health_coach

print("Health Coach Agent imported successfully!")

## Part 1: Simple Mode

The simplest way to use the Health Coach - quick responses with lighter memory updates.

In [ ]:
# Create and configure the coach
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER) # Implicitly uses global config

print("Health Coach initialized in simple mode!")

In [ ]:
# Single turn conversation
response = coach.respond("I've been having trouble sleeping lately")
print("Coach:", response)

In [ ]:
# Continue the conversation
response = coach.respond("I keep waking up around 3am and can't get back to sleep")
print("Coach:", response)

In [ ]:
# Check conversation stats
print("Conversation Stats:")
print(coach.get_stats())

In [ ]:
# View conversation history
print("Conversation History:")
print(coach.get_conversation_history())

## Part 2: Full Conversation Mode

Using the full system prompt with complete memory state tracking.

In [ ]:
# Create a new coach with full mode
coach_full = HealthCoachAgent(simple_mode=False)
coach_full.configure(api_key=API_KEY, provider=PROVIDER)

# Initialize with user context
coach_full.initialize_context("""
User Profile:
- 45 year old male
- Works desk job, 50+ hours/week
- Interested in improving energy levels
""")

print("Full mode coach initialized with user context!")

In [ ]:
# Start a goal-oriented conversation
response = coach_full.respond("I want to have more energy during the day")
print("Coach:", response)

In [ ]:
# The coach will ask clarifying questions
response = coach_full.respond("I usually feel exhausted by 2pm. I drink coffee but it doesn't help anymore.")
print("Coach:", response)

In [ ]:
# Provide more context
response = coach_full.respond("I usually get about 6 hours of sleep. My kids wake me up early.")
print("Coach:", response)

In [ ]:
# Check memory states (what the coach has learned)
print("Memory States:")
print(coach_full.dump_memory())

## Part 3: Responding with External Analysis

The Health Coach can incorporate analysis from other sources (e.g., from an orchestrator that has gathered insights from specialist agents).

In [ ]:
# Create a fresh coach
coach_with_analysis = create_health_coach(
    simple_mode=True
)
coach_with_analysis.configure(api_key=API_KEY, provider=PROVIDER)

# Simulate analysis that would come from other agents
external_analysis = """
Data Science Agent Findings:
- Average sleep duration: 5.8 hours (below recommended 7-9 hours)
- Sleep efficiency: 72% (below optimal 85%)
- Resting heart rate trending up over past month

Domain Expert Insights:
- Sleep duration correlates with reported fatigue
- Consider sleep hygiene improvements before medication
- Rule out sleep apnea given the frequent wakings
"""

# Coach responds incorporating the analysis
response = coach_with_analysis.respond_with_analysis(
    user_message="Why am I always so tired?",
    analysis=external_analysis
)
print("Coach (with analysis):")
print(response)

## Part 4: Custom System Prompt

In [ ]:
# Create coach with custom system prompt
custom_prompt = """
You are a friendly fitness coach specializing in helping busy professionals.
- Focus on practical, time-efficient solutions
- Be encouraging but realistic
- Ask about schedule constraints before making recommendations
- Keep responses concise
"""

custom_coach = HealthCoachAgent(system_prompt=custom_prompt)
custom_coach.configure(api_key=API_KEY, provider=PROVIDER)

response = custom_coach.respond("I want to start exercising but I'm always too busy")
print("Custom Coach:", response)

## Part 5: Conversation Flow Control

The coach automatically detects when to make recommendations or end conversations.

In [ ]:
# Demonstrate conversation ending
coach_ending = HealthCoachAgent(simple_mode=False)
coach_ending.configure(api_key=API_KEY, provider=PROVIDER)

# Have a brief conversation
coach_ending.respond("I want to drink more water")
coach_ending.respond("I usually only drink 2-3 glasses a day")
coach_ending.respond("I'll try keeping a water bottle at my desk")

# Signal end of conversation
response = coach_ending.respond("Thanks, that's really helpful! I'll give it a try.")
print("Coach:", response)

# Check if finish was detected
print("\nFinish detected:", "FINISH" in coach_ending.memory.get("finish_desc", ""))

## Part 6: Interactive Conversation

A simple interactive loop for testing the coach.

In [ ]:
def interactive_chat(coach, max_turns=5):
    """Simple interactive chat loop."""
    print("Starting conversation with Health Coach...")
    print("(Type 'quit' to exit, 'reset' to start over, 'memory' to see memory states)")
    print("-" * 50)
    
    for i in range(max_turns):
        user_input = input(f"\nYou: ")
        
        if user_input.lower() == 'quit':
            print("\nEnding conversation. Goodbye!")
            break
        elif user_input.lower() == 'reset':
            coach.reset_conversation()
            print("\nConversation reset. Let's start fresh!")
            continue
        elif user_input.lower() == 'memory':
            print("\nMemory States:")
            print(coach.dump_memory())
            continue
        
        response = coach.respond(user_input)
        print(f"\nCoach: {response}")
    
    print("\n" + "-" * 50)
    print(f"Conversation ended after {coach.num_turns} turns.")

In [ ]:
# Uncomment to run interactive chat
# coach_interactive = create_health_coach(
#     simple_mode=True
# )
# coach_interactive.configure(api_key=API_KEY, provider=PROVIDER)
# interactive_chat(coach_interactive)

## Summary

The Health Coach Agent provides:

1. **Conversational Interface**: Natural multi-turn health discussions
2. **Memory Management**: Tracks user goals, constraints, and profile
3. **Smart Flow Control**: Automatic recommendation and finish detection
4. **Analysis Integration**: Can incorporate insights passed from other agents
5. **Customization**: Custom system prompts and conversation context

### Next Steps

- See `05_full_pipeline.ipynb` for how the orchestrator coordinates all agents
- Customize prompts in `pha/prompts/health_coach_prompts.py`